# Task 2
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [1]:
import numpy as np
import pandas as pd
# Add any other imports you need here
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, RBF, Matern, RationalQuadratic

# test
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.gaussian_process.kernels import WhiteKernel
from sklearn.gaussian_process.kernels import ConstantKernel as C

# Data Loading
TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test
(and potentially change initialization of variables to accomodate how you deal with non-numeric data)

In [2]:
"""
This loads the training and test data, preprocesses it, removes the NaN
values and interpolates the missing data using imputation

Parameters
----------
Compute
----------
X_train: matrix of floats, training input with features
y_train: array of floats, training output with labels
X_test: matrix of floats: dim = (100, ?), test input with features
"""
# Load training data
train_df = pd.read_csv("train.csv")
    
print("Training data:")
print("Shape:", train_df.shape)
print(train_df.head(2))
print('\n')
    
# Load test data
test_df = pd.read_csv("test.csv")

print("Test data:")
print(test_df.shape)
print(test_df.head(2))

# Dummy initialization of the X_train, X_test and y_train   
# TODO: Depending on how you deal with the non-numeric data, you may want to 
# modify/ignore the initialization of these variables
"""
X_train = np.zeros_like(train_df.drop(['price_CHF'],axis=1))
y_train = np.zeros_like(train_df['price_CHF'])
X_test = np.zeros_like(test_df)
"""

# TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test

# remove rows that done have a CHF price so we dont train on made up data
train_df = train_df.dropna(subset=['price_CHF'])

# preprocessing using pandas get_dummies function
train_df = pd.get_dummies(train_df, columns=['season'])
test_df = pd.get_dummies(test_df, columns=['season'])

# proper initialization of X_train, X_test and y_train
X_train = train_df.drop(['price_CHF'],axis=1)
#X_train_season = X_train['season']              # TEMP
#X_train = X_train.drop(['season'],axis=1)       # TEMP
y_train = train_df['price_CHF']
X_test = test_df
#X_test_season = X_test['season']                # TEMP
#X_test = X_test.drop(['season'],axis=1)         # TEMP

#print("train_df", X_train.head(5))
#print("test_df", X_test.head(5))

# imputation time using the median
#imp = SimpleImputer(missing_values=np.nan, strategy='median')

imp = IterativeImputer(max_iter=10, random_state=0)


X_train = imp.fit_transform(X_train)
X_test = imp.transform(X_test)
#print(X_train)


assert (X_train.shape[1] == X_test.shape[1]) and (X_train.shape[0] == y_train.shape[0]) and (X_test.shape[0] == 100), "Invalid data shape"

Training data:
Shape: (900, 11)
   season  price_AUS  price_CHF  price_CZE  price_GER  price_ESP  price_FRA  \
0  spring  -3.348808        NaN  -3.597534  -4.102160  -2.201652  -2.806995   
1  summer  -3.421345  -1.455502  -3.597649  -3.675204        NaN  -2.440406   

   price_UK  price_ITA  price_POL  price_SVK  
0       NaN   -3.61728  -2.758448        NaN  
1 -2.379524        NaN        NaN   -3.72506  


Test data:
(100, 10)
   season  price_AUS  price_CZE  price_GER  price_ESP  price_FRA  price_UK  \
0  spring  -1.504285  -1.632302  -2.347618        NaN        NaN -3.437325   
1  summer  -1.779837  -1.750216  -2.407555  -1.875685        NaN       NaN   

   price_ITA  price_POL  price_SVK  
0  -3.505886  -2.042408        NaN  
1  -3.528359  -2.131659  -2.911154  


# Modeling and Prediction
TODO: Define the model and fit it using training data. Then, use test data to make predictions

In [3]:
"""
This defines the model, fits training data and then does the prediction
with the test data 

Parameters
----------
X_train: matrix of floats, training input with 10 features
y_train: array of floats, training output
X_test: matrix of floats: dim = (100, ?), test input with 10 features

Compute
----------
y_test: array of floats: dim = (100,), predictions on test set
"""
class Model(object):
    def __init__(self):
        super().__init__()
        self._x_train = None
        self._y_train = None
        # store the best regressor here
        self.best_reg = None

    def fit(self, X_train: np.ndarray, y_train: np.ndarray):
        #TODO: Define the model and fit it using (X_train, y_train)
        self._x_train = X_train
        self._y_train = y_train

        # We put all the kernels as candidates and then check which one gives the best result
        candidates = [
            C(1.0, (1e-3, 1e3)) + DotProduct() + WhiteKernel(noise_level=0.1),   # linear kernel
            C(1.0, (1e-3, 1e3)) + RBF(length_scale=1.0) + WhiteKernel(noise_level=0.1),  # squared exponential kernel
            C(1.0, (1e-3, 1e3)) + Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=0.1),
            C(1.0, (1e-3, 1e3)) + RationalQuadratic(length_scale=1.0, alpha=0.1) + WhiteKernel(noise_level=0.1)
        ]

        best_lml = -np.inf

        # Loop over the candidates to see which kernel is the best
        
        for kernel in candidates:
            gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5)
            gpr.fit(X_train, y_train)

            lml = gpr.log_marginal_likelihood(gpr.kernel_.theta)

            if lml > best_lml:
                best_lml = lml
                self.best_reg = gpr
        print(self.best_reg)
        """
        gpr = GaussianProcessRegressor(kernel=Matern(), n_restarts_optimizer=3, normalize_y=True)
        gpr.fit(X_train, y_train)
        self.best_reg = gpr
        """


    def predict(self, X_test: np.ndarray) -> np.ndarray:
        #TODO: Use the model to make predictions y_pred using test data X_test
        # y_pred=np.zeros(X_test.shape[0])
        y_pred = self.best_reg.predict(X_test)
        assert y_pred.shape == (X_test.shape[0],), "Invalid data shape"
        return y_pred

In [4]:
model = Model()
# Use this function to fit the model
model.fit(X_train=X_train, y_train=y_train)
# Use this function for inference
y_pred = model.predict(X_test)
print(y_pred)

GaussianProcessRegressor(kernel=1**2 + RBF(length_scale=1) + WhiteKernel(noise_level=0.1),
                         n_restarts_optimizer=5)
[ 5.75451622  4.04675396  4.08517715  4.55650881  3.76880339  3.4038666
  2.43252035  2.16601687  2.12222532  0.28537118  0.07349348 -0.38884218
 -0.80657706 -1.80260139 -1.8783544  -2.57599486 -1.37828612 -2.38912351
 -1.81688279 -2.29841139 -2.08316055 -1.68522478 -1.84749992 -1.34855772
 -0.97589413 -1.23626172 -1.27130199 -1.2452841  -0.3974094  -0.70750279
 -0.90456591 -1.30853475 -0.4326644  -1.64719656 -0.61878731 -0.66003706
 -1.29554728 -1.48616044 -1.57329982 -1.60582495 -2.18891985 -1.62799117
 -1.63807011 -1.45790027 -1.80059557 -1.83445105 -1.45007578 -1.29502795
 -1.23399537 -1.18430379 -1.38490755 -1.2447213  -1.42840996 -1.65871835
 -1.37626888 -1.07850953 -1.25980754 -1.34186763 -1.46870862 -0.9129977
 -1.14570816 -1.26296325 -1.01480919 -1.48439942 -1.27266183 -1.42508905
 -0.87314342 -1.60912783 -1.55310213 -2.90572347 -2.2149596

c:\Users\lmich\anaconda3\envs\IML\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__alpha is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


# Saving Results
You don't have to change this

In [5]:
dt = pd.DataFrame(y_pred) 
dt.columns = ['price_CHF']
dt.to_csv('results.csv', index=False)
print("\nResults file successfully generated!")


Results file successfully generated!
